# Kaggle Notebook 02 — Optional Gemini Generation Demo

Run this notebook on Kaggle only if you want to demonstrate generation tasks:

- Q&A
- information extraction
- retrieval-based summarization and full-document map-reduce summarization
- quiz generation
- flashcard generation

This notebook needs `GOOGLE_API_KEY` from Kaggle Secrets or environment variables.

In [ ]:
%pip install -q pymupdf rank-bm25 sentence-transformers faiss-cpu google-generativeai pandas

In [ ]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    pass
elif (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif Path("/kaggle/working/final-notebooklm-study-assistant").exists():
    PROJECT_ROOT = Path("/kaggle/working/final-notebooklm-study-assistant")
else:
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "kaggle_generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle Secret support. Create a secret named GOOGLE_API_KEY in Kaggle.
try:
    from kaggle_secrets import UserSecretsClient
    secret_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    if secret_key:
        os.environ["GOOGLE_API_KEY"] = secret_key
except Exception as exc:
    print("Kaggle secret not loaded. You can still set os.environ['GOOGLE_API_KEY'] manually.", exc)

print("Has GOOGLE_API_KEY:", bool(os.getenv("GOOGLE_API_KEY")))

In [ ]:
from study_assistant.retrieval import StudyIndex
from study_assistant.generation import (
    GeminiClient,
    answer_question,
    extract_information,
    summarize_topic,
    summarize_document,
    retrieved_from_chunks,
    generate_quiz,
    generate_flashcards,
    result_to_jsonable,
)

index = StudyIndex()
index.build_from_data_dir(DATA_DIR)
client = GeminiClient(api_key=os.getenv("GOOGLE_API_KEY", ""))
print(f"Indexed {len(index.chunks)} chunks")

In [ ]:
question = "What is the default online retrieval pipeline?"
chunks = index.retrieve_hybrid(question, k=5, alpha=0.30)
answer = answer_question(question, chunks, client)
print(answer.answer)
json.dump(result_to_jsonable(answer), open(OUTPUT_DIR / "answer.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

In [ ]:
request = "Extract the main retrieval components and their purposes."
chunks = index.retrieve_hybrid(request, k=5, alpha=0.30)
extracted = extract_information(request, chunks, client)
print(extracted.answer)
json.dump(result_to_jsonable(extracted), open(OUTPUT_DIR / "extraction.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

In [ ]:
topic = "hybrid retrieval and evaluation"
chunks = index.retrieve_hybrid(topic, k=8, alpha=0.30)
summary = summarize_topic(topic, chunks, client)
print(summary.summary)
print(summary.key_points)
json.dump(result_to_jsonable(summary), open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

## Full-document map-reduce summary


In [ ]:
all_chunks = retrieved_from_chunks(index.chunks)
full_summary = summarize_document("main concepts", all_chunks, client)
print(full_summary.summary)
print(full_summary.key_points)
json.dump(result_to_jsonable(full_summary), open(OUTPUT_DIR / "full_document_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)


In [ ]:
topic = "RAG pipeline"
chunks = index.retrieve_hybrid(topic, k=5, alpha=0.30)
quiz = generate_quiz(topic, chunks, count=5, client=client)
for i, item in enumerate(quiz.items, 1):
    print(f"Q{i}. {item.question}")
    print(item.options)
    print("answer:", item.correct_index, "|", item.explanation)
    print()
json.dump(result_to_jsonable(quiz), open(OUTPUT_DIR / "quiz.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

In [ ]:
topic = "retrieval metrics"
chunks = index.retrieve_hybrid(topic, k=5, alpha=0.30)
flashcards = generate_flashcards(topic, chunks, count=8, client=client)
for card in flashcards.cards:
    print("Front:", card.front)
    print("Back:", card.back)
    print()
json.dump(result_to_jsonable(flashcards), open(OUTPUT_DIR / "flashcards.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

In [ ]:
list(OUTPUT_DIR.glob("*.json"))